# 04 — SOLOv2 (mmdetection) — different architecture family

ทดลองข้ามสถาปัตยกรรม: **SOLOv2 R50-FPN ms-3x** (dynamic-kernel instance segmentation)
vs baseline YOLO26n-seg (`arm_gold-2`).

- env: `.venv_mmdet` (torch 2.1.2+cu121, mmcv 2.1.0 compiled, mmdet 3.3.0)
- data: `data/gold` → COCO JSON (polygon seg), 3 splits
- train: solov2_r50_fpn_ms-3x recipe, batch 4, lr 0.0025 (scaled จาก bs16), seed 42
- eval: **custom mask-AP harness เดียวกัน** (strict + sieved + TP/FP/FN @0.5) → เทียบ baseline ตรง ๆ

In [1]:
import numpy as np, cv2, json, glob, time
import pandas as pd
from pathlib import Path

WORK = Path('.').resolve()          # experiment/
VENV = Path.home()/'YOLO AGGREGATE/.venv_mmdet'
CFG_DIR = VENV/'lib/python3.11/site-packages/mmdet/.mim/configs/solov2'
CKPT = Path.home()/'YOLO AGGREGATE/ckpts/solov2_r50_3x_coco.pth'
DS = WORK/'data/gold'
NAMES = ['gravel','sand']
print('cfg dir:', CFG_DIR.exists(), '| ckpt:', CKPT.exists(), '| ds:', DS.exists())

cfg dir: True | ckpt: True | ds: True


In [2]:
# --- convert gold (YOLO-seg) -> COCO json for train/valid/test ---
def yolo2coco(split):
    img_dir = DS/split/'images'; lbl_dir = DS/split/'labels'
    coco = dict(images=[], annotations=[], categories=[
        dict(id=1,name='gravel'), dict(id=2,name='sand')])
    aid = 0
    for i, ip in enumerate(sorted(img_dir.glob('*.jpg')), 1):
        im = cv2.imread(str(ip)); H, W = im.shape[:2]
        coco['images'].append(dict(id=i, file_name=f'{split}/images/{ip.name}',
                                   width=W, height=H))
        lf = lbl_dir/(ip.stem+'.txt')
        if not lf.exists(): continue
        for ln in lf.read_text().strip().splitlines():
            p = ln.split()
            if len(p) < 7: continue
            xy = (np.array(p[1:], float).reshape(-1,2) * [W,H])
            seg = xy.ravel().tolist()
            x0,y0 = xy[:,0].min(), xy[:,1].min()
            bw,bh = xy[:,0].max()-x0, xy[:,1].max()-y0
            m = np.zeros((H,W),np.uint8); cv2.fillPoly(m,[xy.astype(np.int32)],1)
            coco['annotations'].append(dict(
                id=aid, image_id=i, category_id=int(p[0])+1,
                segmentation=[seg], bbox=[x0,y0,bw,bh],
                area=float(m.sum()), iscrowd=0))
            aid += 1
    out = DS/f'{split}.json'
    out.write_text(json.dumps(coco))
    print(split, len(coco['images']), 'imgs,', aid, 'anns ->', out.name)
for s in ('train','valid','test'): yolo2coco(s)

train 753 imgs, 2822 anns -> train.json


valid 226 imgs, 691 anns -> valid.json
test 105 imgs, 348 anns -> test.json


In [3]:
# --- build config ---
from mmengine.config import Config
cfg = Config.fromfile(str(CFG_DIR/'solov2_r50_fpn_ms-3x_coco.py'))
cfg.model.mask_head.num_classes = 2
classes = dict(classes=tuple(NAMES))
for dl_name, split in (('train_dataloader','train'),
                       ('val_dataloader','valid'),
                       ('test_dataloader','test')):
    dl = getattr(cfg, dl_name)
    dl.dataset.data_root = str(DS)
    dl.dataset.ann_file = f'{split}.json'
    dl.dataset.data_prefix = dict(img='')
    dl.dataset.metainfo = classes
cfg.train_dataloader.batch_size = 2
cfg.train_dataloader.num_workers = 4
cfg.optim_wrapper.optimizer.lr = 0.00125   # bs16->bs4 linear scale
cfg.load_from = str(CKPT)
cfg.work_dir = str(WORK/'runs/solov2_r50_3x')
cfg.randomness = dict(seed=42, deterministic=False)
cfg.val_evaluator.ann_file = str(DS/'valid.json')
cfg.val_evaluator.metric = ['bbox','segm']
cfg.test_evaluator.ann_file = str(DS/'test.json')
cfg.test_evaluator.metric = ['bbox','segm']
cfg.default_hooks.checkpoint.save_best = 'coco/segm_mAP'
cfg.default_hooks.checkpoint.rule = 'greater'
cfg.dump(str(WORK/'solov2_gold.py'))
print('epochs:',cfg.train_cfg.max_epochs,'| batch:',cfg.train_dataloader.batch_size,
      '| lr:',cfg.optim_wrapper.optimizer.lr,'| work_dir:',cfg.work_dir)

epochs: 36 | batch: 2 | lr: 0.00125 | work_dir: /home/supawich/Desktop/Aggregate_Project/experiment/runs/solov2_r50_3x


In [4]:
# --- train (36 epochs ms-3x) ---
from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
t0 = time.time()
runner.train()
print(f'train done in {(time.time()-t0)/60:.1f} min')

10/09 22:07:21 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.11.14 (main, Feb 12 2026, 00:42:50) [Clang 21.1.4 ]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA GeForce RTX 3080 Ti
    CUDA_HOME: None
    GCC: cc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.1.2+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code=sm

10/09 22:07:21 - mmengine - INFO - Config:
auto_scale_lr = dict(base_batch_size=16, enable=False)
backend_args = None
data_root = 'data/coco/'
dataset_type = 'CocoDataset'
default_hooks = dict(
    checkpoint=dict(
        interval=1,
        rule='greater',
        save_best='coco/segm_mAP',
        type='CheckpointHook'),
    logger=dict(interval=50, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=dict(type='DetVisualizationHook'))
default_scope = 'mmdet'
env_cfg = dict(
    cudnn_benchmark=False,
    dist_cfg=dict(backend='nccl'),
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0))
load_from = '/home/supawich/YOLO AGGREGATE/ckpts/solov2_r50_3x_coco.pth'
log_level = 'INFO'
log_processor = dict(by_epoch=True, type='LogProcessor', window_size=50)
max_epochs = 36
model = dict(
    backbone=dict(
        depth=50,
        frozen_stages=1,
        

10/09 22:07:22 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.


10/09 22:07:22 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(BELOW_NORMAL) LoggerHook                         
(LOW         ) ParamSchedulerHook                 
(VERY_LOW    ) CheckpointHook

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


...

Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*


DONE (t=0.14s).
Accumulating evaluation results...


DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.526
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=1000 ] = 0.723
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=1000 ] = 0.572
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.067
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.458
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.669
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.633
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=300 ] = 0.633
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=1000 ] = 0.633
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1000 ] = 0.135
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=1000 ] = 0.586
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=1000 ] = 0.756
10/09 23:00:00 

10/09 23:00:00 - mmengine - INFO - Epoch(val) [36][226/226]    coco/bbox_mAP: 0.0000  coco/bbox_mAP_50: 0.0000  coco/bbox_mAP_75: 0.0000  coco/bbox_mAP_s: 0.0000  coco/bbox_mAP_m: 0.0000  coco/bbox_mAP_l: 0.0000  coco/segm_mAP: 0.5260  coco/segm_mAP_50: 0.7230  coco/segm_mAP_75: 0.5720  coco/segm_mAP_s: 0.0670  coco/segm_mAP_m: 0.4580  coco/segm_mAP_l: 0.6690  data_time: 0.0020  time: 0.0445


train done in 52.6 min


In [5]:
# --- predict test set -> cache preds ---
import torch
from mmdet.apis import init_detector, inference_detector
best = sorted((Path(cfg.work_dir)).glob('best_coco_segm_mAP*.pth'))
best = best[-1] if best else Path(cfg.work_dir)/'epoch_36.pth'
print('ckpt:', best.name)
mdl = init_detector(str(WORK/'solov2_gold.py'), str(best), device='cuda')
mdl.cfg.model.test_cfg.score_thr = 0.01   # keep low-conf for our own eval
CACHE_DIR = WORK/'results/preds_cache'; CACHE_DIR.mkdir(exist_ok=True)
GT={}; SHW={}
TI = DS/'test/images'; TL = DS/'test/labels'
AREA_MIN, ASP = 0.005, 8.0
def load_gt(lf,W,H):
    out=[]
    if lf.exists():
        for ln in lf.read_text().strip().splitlines():
            p=ln.split()
            if len(p)<7: continue
            xy=np.array(p[1:],float).reshape(-1,2); px=(xy*[W,H]).astype(np.int32)
            m=np.zeros((H,W),np.uint8); cv2.fillPoly(m,[px],1)
            bw,bh=np.ptp(px[:,0]),np.ptp(px[:,1])
            out.append(dict(cls=int(p[0]),mask=m.astype(bool),area=float(m.mean()),
                            aspect=float(max(bw,bh)/max(1,min(bw,bh)))))
    return out
def sieve(g): return g['area']>=AREA_MIN and g['aspect']<=ASP
for ip in sorted(TI.glob('*.jpg')):
    im=cv2.imread(str(ip)); H,W=im.shape[:2]; SHW[ip.name]=(W,H)
    GT[ip.name]=load_gt(TL/(ip.stem+'.txt'),W,H)
PR={}
for nm,(W,H) in SHW.items():
    r = inference_detector(mdl, str(TI/nm))
    p = r.pred_instances
    pr=[]
    for c,cf,m in zip(p.labels.cpu().numpy(), p.scores.cpu().numpy(),
                      p.masks.cpu().numpy()):
        mm = cv2.resize(m.astype(np.uint8),(W,H))>0.5 if m.shape!=(H,W) else m.astype(bool)
        if mm.any(): pr.append(dict(cls=int(c),conf=float(cf),mask=mm))
    PR[nm]=pr
np.savez_compressed(CACHE_DIR/'solov2_test_preds.npz',
                    names=np.array(list(PR)),
                    **{f'k{i}':dict(cls=np.array([p['cls'] for p in pr]),
                                   conf=np.array([p['conf'] for p in pr]),
                                   masks=np.stack([np.packbits(p['mask'],axis=None)
                                                   for p in pr]) if pr else np.zeros(0,np.uint8),
                                   n=np.array([len(pr)]))
                       for i,(nm,pr) in enumerate(PR.items())})
print('preds cached:', sum(len(v) for v in PR.values()), 'insts')

ckpt: best_coco_segm_mAP_epoch_10.pth


Loads checkpoint by local backend from path: /home/supawich/Desktop/Aggregate_Project/experiment/runs/solov2_r50_3x/best_coco_segm_mAP_epoch_10.pth


preds cached: 1282 insts


In [6]:
# --- same eval harness as nb01/nb03 ---
IOUS = np.round(np.arange(0.5,1.0,0.05),2)
def iou(a,b):
    u=(a|b).sum(); return (a&b).sum()/u if u else 0.
def evaluate(PR, sieved=True, conf_op=0.25):
    aps=[]
    for cls in (0,1):
        n_gt=sum(1 for im in GT.values() for g in im
                 if g['cls']==cls and (not sieved or sieve(g)))
        aa=[]
        for thr in IOUS:
            recs=[]
            for nm in GT:
                keep=[g for g in GT[nm] if g['cls']==cls and (not sieved or sieve(g))]
                used=set()
                for p in sorted([p for p in PR[nm] if p['cls']==cls
                                 and (not sieved or p['mask'].mean()>=AREA_MIN)],
                                key=lambda x:-x['conf']):
                    best,bj=0.,-1
                    for j,g in enumerate(keep):
                        if j in used: continue
                        v=iou(p['mask'],g['mask'])
                        if v>best: best,bj=v,j
                    if best>=thr: used.add(bj); recs.append((p['conf'],1))
                    else: recs.append((p['conf'],0))
            recs.sort(key=lambda x:-x[0])
            tp=np.cumsum([r[1] for r in recs]); fp=np.cumsum([1-r[1] for r in recs])
            rec=tp/max(n_gt,1); prec=np.divide(tp,np.maximum(tp+fp,1))
            mrec=np.concatenate([[0],rec,[1]]); mpre=np.concatenate([[0],prec,[0]])
            for i in range(len(mpre)-2,-1,-1): mpre[i]=max(mpre[i],mpre[i+1])
            aa.append(mpre[np.minimum(np.searchsorted(mrec,np.linspace(0,1,101),'left'),
                                     len(mpre)-1)].mean())
        aps.append(float(np.nanmean(aa)))
    tp=fp=fn=0
    for nm in GT:
        used=set()
        keep=[g for g in GT[nm] if not sieved or sieve(g)]
        for p in sorted([p for p in PR[nm] if p['conf']>=conf_op
                         and (not sieved or p['mask'].mean()>=AREA_MIN)],
                        key=lambda x:-x['conf']):
            best,bj=0.,-1
            for j,g in enumerate(keep):
                if j in used or g['cls']!=p['cls']: continue
                v=iou(p['mask'],g['mask'])
                if v>best: best,bj=v,j
            if best>=0.5: used.add(bj); tp+=1
            else: fp+=1
        fn+=len(keep)-len(used)
    return dict(mAP=round(np.nanmean(aps),4),gravel=round(aps[0],4),
                sand=round(aps[1],4),TP=tp,FP=fp,FN=fn)

s=evaluate(PR,sieved=False); v=evaluate(PR,sieved=True)
res=pd.DataFrame([
    dict(model='YOLO26n-seg baseline', strict=0.3548, sieved=0.4485, FN=114, FP=43),
    dict(model='SOLOv2-R50 ms-3x', strict=s['mAP'], sieved=v['mAP'],
         FN=v['FN'], FP=v['FP'], gravel_strict=s['gravel'], sand_strict=s['sand'])])
res.to_csv(WORK/'results/solov2_eval.csv',index=False)
print(res.to_string(index=False))

               model  strict  sieved  FN  FP  gravel_strict  sand_strict
YOLO26n-seg baseline  0.3548  0.4485 114  43            NaN          NaN
    SOLOv2-R50 ms-3x  0.3447  0.4236 107 104         0.5206       0.1688
